# Purpose:
- Curate all the datasets collected in lims for GCaMP8 characterization
- Document error data, and why
- Compare with what's in the codeocean
    - Requires a different environment (because allensdk is limited in python version)
- Follow up from 240701_query_lims_data.ipynb

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import h5py
import os

from brain_observatory_qc.data_access import from_lims

from allensdk.brain_observatory.behavior.behavior_project_cache import VisualBehaviorOphysProjectCache as bpc

from pymongo import MongoClient
mongo = MongoClient("flaskapp.corp.alleninstitute.org", 27017)


In [3]:
cache = bpc.from_lims()
table = cache.get_ophys_experiment_table(passed_only=False)

# Helper functions

In [4]:
# helper function
def get_session_info_per_group_of_mice(mouse_ids, gcamp_type):
    experiment_table = pd.DataFrame()
    problem_experiments = []
    for mouse_id in mouse_ids:
        mouse_data = from_lims.get_imaging_ids_for_mouse_id(mouse_id)
        if len(mouse_data) > 0:
            ophys_experiment_ids = mouse_data.ophys_experiment_id.values
            for ophys_experiment_id in ophys_experiment_ids: 
                try: 
                    expt_info = from_lims.get_general_info_for_ophys_experiment_id(ophys_experiment_id)
                    if len(expt_info) > 0:
                        expt_info = expt_info.iloc[[0]]
                        genotype = from_lims.get_genotype_for_ophys_experiment_id(ophys_experiment_id)
                        expt_info["mouse_id"] = mouse_id # expt_info only contains `donor_id` and `specimen_id`, need to add mouse_id here
                        expt_info["full_genotype"] = genotype.full_genotype.values[0] # base function doesnt get genotype, add it
                        experiment_table = pd.concat([experiment_table, expt_info])
                except: 
                    # print('problem for mouse_id: ', mouse_id, ', expt_id: ', ophys_experiment_id)
                    problem_experiments.append(ophys_experiment_id)
    experiment_table['gcamp'] = gcamp_type
    return experiment_table, problem_experiments


def get_session_info_per_group_of_sessions(session_ids, mouse_ids, gcamp_type):
    experiment_table = pd.DataFrame()
    problem_experiments = []
    for session_id, mouse_id in zip(session_ids, mouse_ids):
        ophys_experiment_ids = from_lims.get_ophys_experiment_ids_for_ophys_session_id(session_id).ophys_experiment_id.values
        for ophys_experiment_id in ophys_experiment_ids: 
            try: 
                expt_info = from_lims.get_general_info_for_ophys_experiment_id(ophys_experiment_id)
                if len(expt_info) > 0:
                    expt_info = expt_info.iloc[[0]]
                    genotype = from_lims.get_genotype_for_ophys_experiment_id(ophys_experiment_id)
                    expt_info["mouse_id"] = mouse_id # expt_info only contains `donor_id` and `specimen_id`, need to add mouse_id here
                    expt_info["full_genotype"] = genotype.full_genotype.values[0] # base function doesnt get genotype, add it
                    experiment_table = pd.concat([experiment_table, expt_info])
            except: 
                # print('problem for mouse_id: ', mouse_id, ', expt_id: ', ophys_experiment_id)
                problem_experiments.append(ophys_experiment_id)
    experiment_table['gcamp'] = gcamp_type
    return experiment_table, problem_experiments


def get_mongo_client(username, password, host, port):
    client = MongoClient(f'mongodb://{username}:{password}@{host}:{port}')
    return client


def get_zdrift(opid):
    try:
        client = get_mongo_client('public', 'public_password', 'qc-sys-db', 27017)
    except:
        raise('Failed to connect to mongo')
    
    opid = int(opid)
    try:
        record = client.records.metrics.find_one({'data_id': opid})
    except:
        raise('No record found for opid: ', opid)
        
    try:
        zdrift = record['local_z_stack']['z_drift_corr_um_diff']
    except:
        zdrift = np.nan
    return zdrift

def get_intensity_drift(opid):
    try:
        client = get_mongo_client('public', 'public_password', 'qc-sys-db', 27017)
    except:
        raise('Failed to connect to mongo')
    
    opid = int(opid)
    try:
        record = client.records.metrics.find_one({'data_id': opid})
    except:
        raise('No record found for opid: ', opid)
        
    try:
        intensity_drift = record['motion_corr_physio']['percent_change_intensity']
    except:
        intensity_drift = np.nan
    return intensity_drift


def get_monitor_sync(osid):
    try:
        client = get_mongo_client('public', 'public_password', 'qc-sys-db', 27017)
    except:
        raise('Failed to connect to mongo')
    
    osid = int(osid)
    try:
        record = client.records.metrics.find_one({'data_id': osid})
    except:
        raise('No record found for osid: ', osid)
        
    try:
        monitor_sync = record['rig_sync']['display_lag']
    except:
        monitor_sync = np.nan
    return monitor_sync

# Using from_lims_utilities
- updated version (currently in from_lims_update branch; 07/10/2024)

# Manual log of mouse IDs
- or session ID if specific session is preferred (when there are multiple, use the one with the lowest zdrift failure planes)

In [5]:
mids_ribo_aav_local = [726087, 719363] # 719364 removed due to brain health issue
# mids_ribo_aav_ro = [730929, 730932, 730933] # Note: all of these has reduced signal after some sessions
mids_ribo_aav_icv = [738331, 738332, 738333]
mids_snap25_oi4_dox = [726433, 747443, 755212, 759075]
mids_slc32a1_oi4 = [724567, 729088, 753561, 753562, 758265, ]
mids_slc32a1_oi1 = [687000, 693996, 692478, 687001, 754804, 757436] # need to exclude osid 1303235340, 1299462513
# mids_slc17a7_oi1_dox = [733794, 750451, 750447] # 750451 and 750447 were not imaged. Ignore this line
# mids_cux2_oi1 = [735169, 735170] # Ignore this for now as well. Too sparse
mids_oi4_homo = [741863, 741865, 759732, 759731, 759730]
# mids_oi1_icre = [762818, 762819, 762821] # Astrocytic expression

group_mid_dict = {'slc32a1_oi1': mids_slc32a1_oi1,
                  'slc32a1_oi4': mids_slc32a1_oi4,
                  'snap25_oi4_dox': mids_snap25_oi4_dox,
                  'oi4_homo': mids_oi4_homo,
                  'ribo_aav_local': mids_ribo_aav_local,
                  'ribo_aav_icv': mids_ribo_aav_icv,
}

remove_session_ids = [1303235340, 1299462513, 1369518919, 1370949854, 1371209490]  
# 1303235340 was a test session. 1299462513 had very low signal (don't know why)
# 1369518919 was motion correction test
# 1370949854, 1371209490 were early sessions where 1x4 were collected instead of 1x2
remove_acquisition_dates = ['2024-08-14', '2024-09-24']  # weird power setting

In [6]:
gcamp_table_pre = pd.DataFrame()
problems_table = []
for key, val in group_mid_dict.items():
    temp_table, temp_problems = get_session_info_per_group_of_mice(val, key)
    gcamp_table_pre = pd.concat([gcamp_table_pre, temp_table])
    problems_table.extend(temp_problems)

In [7]:
problems_table

[]

In [8]:
# Get sessions with target session types
target_session_types = ['STAGE_1', 'OPHYS_2_images_A_passive']
gcamp_table_pre = gcamp_table_pre[gcamp_table_pre.session_type.isin(target_session_types)].copy()

# Manual removal of erraneous sessions
gcamp_table_pre = gcamp_table_pre[~gcamp_table_pre.ophys_session_id.isin(remove_session_ids)].copy()
assert gcamp_table_pre.ophys_session_id.isna().sum() == 0
assert np.array_equal(gcamp_table_pre.session_type.unique(), np.array(['STAGE_1', 'OPHYS_2_images_A_passive']))


In [9]:
depth_divider = [0, 125, 225, 325, 500]
depth_labels = [75, 175, 275, 375]
gcamp_table_pre['target_depth'] = pd.cut(gcamp_table_pre.depth, bins=depth_divider, labels=depth_labels, right=False)


In [11]:
# post-processing

# check if all gcamp8 data has 2 planes
gcamp_table = gcamp_table_pre.copy()

plane_counts = gcamp_table.groupby('ophys_session_id').count().ophys_experiment_id
pilot_osids = gcamp_table.ophys_session_id.unique()
pilot_i = plane_counts.index.isin(pilot_osids)
error_session_ids = plane_counts.iloc[pilot_i].index.values[np.where(plane_counts.iloc[pilot_i].values != 2)]
print(f'Num error sessions due to num planes: {len(error_session_ids)}')
print(f'Removing {error_session_ids}')
gcamp_table = gcamp_table.query('ophys_session_id not in @error_session_ids')

# Remove those taken with weird power setting
error_power_sessions = gcamp_table[(gcamp_table.date_of_acquisition >= remove_acquisition_dates[0]) &
                          (gcamp_table.date_of_acquisition <= remove_acquisition_dates[1])].ophys_session_id.unique()
gcamp_table = gcamp_table[~gcamp_table.ophys_session_id.isin(error_power_sessions)].copy()

num_sessions = gcamp_table.ophys_session_id.nunique()
print(f'\nPower error sessions: {len(error_power_sessions)}')
print(f'Total number of sessions: {num_sessions}\n')

# ids into int
id_columns = ['ophys_experiment_id', 'ophys_session_id', 'mouse_id']
gcamp_table[id_columns] = gcamp_table[id_columns].astype(int)

# sort by gcamp
gcamp_table = gcamp_table.sort_values('gcamp')

# Add zdrift QC metric (from mouse-seeks)
gcamp_table['zdrift'] = gcamp_table.ophys_experiment_id.apply(get_zdrift)

# Add monitor sync QC (from mouse-seeks)
gcamp_table['monitor_sync'] = gcamp_table.ophys_session_id.apply(get_monitor_sync)

# # Remove sesions with error in zdrift calculation
# error_osids = gcamp_table.query('zdrift.isna()').ophys_session_id.unique()
# gcamp_table = gcamp_table[~gcamp_table.ophys_session_id.isin(error_osids)].copy()
# gcamp_table['abs_zdrift'] = np.abs(gcamp_table.zdrift)
# There are some sessions where z-drift calculation failed, because of local z-stack processing failure
# But these still have the local z-stack data, so we can calculate the z-drift in codeocean

# add intensity drift
gcamp_table['intensity_drift'] = gcamp_table.ophys_experiment_id.apply(get_intensity_drift)

# Select columns to save
info_columns = ['ophys_experiment_id', 'ophys_session_id', 'gcamp', 'mouse_id', 'date_of_acquisition', 'session_storage_directory',
                'depth', 'targeted_structure', 'session_type', 'full_genotype', 'zdrift', 'intensity_drift', 'monitor_sync',
                'equipment_name', 'specimen_storage_directory', 'project']
gcamp_info = gcamp_table[info_columns].set_index('ophys_experiment_id').drop_duplicates()

# changing date_of_acquisition type to show up correctly in csv
gcamp_info.date_of_acquisition = gcamp_info.date_of_acquisition.dt.strftime('%Y-%m-%d_%H-%M-%S')

# session-based table for uploading
session_info_column = ['gcamp', 'full_genotype', 'mouse_id', 'date_of_acquisition', 'session_type', 'session_storage_directory', 'project']
gcamp_session_info = gcamp_info.copy().set_index('ophys_session_id', drop=True)[session_info_column].drop_duplicates()

print(len(gcamp_info))
print(len(gcamp_session_info))

Num error sessions due to num planes: 0
Removing []

Power error sessions: 15
Total number of sessions: 97

194
97


In [12]:
# How many session from each group, for each session type, per depth?
# Set display options to show all columns and rows
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# View the dataframe
gcamp_table.groupby(['gcamp', 'session_type', 'target_depth']).size().reset_index(name='counts').pivot(index=('gcamp', 'target_depth'), columns='session_type', values='counts').fillna(0).astype(int).T

gcamp                    oi4_homo             ribo_aav_icv              \
target_depth                   75 175 275 375           75 175 275 375   
session_type                                                             
OPHYS_2_images_A_passive        0   0   0   0            2   2   2   2   
STAGE_1                         4   5   5   4            2   3   3   2   

gcamp                    ribo_aav_local             slc32a1_oi1              \
target_depth                         75 175 275 375          75 175 275 375   
session_type                                                                  
OPHYS_2_images_A_passive              4   4   4   4           5   5   5   5   
STAGE_1                               4   4   4   4          10  10  10  10   

gcamp                    slc32a1_oi4             snap25_oi4_dox              
target_depth                      75 175 275 375             75 175 275 375  
session_type                                                                 
OPHYS_2_images_A_passive           3   5   5   3              4   4   4   4  
STAGE_1                            4   3   3   4              5   5   5   5

In [13]:
gcamp_table.groupby('session_type').size()

session_type
OPHYS_2_images_A_passive     76
STAGE_1                     118
dtype: int64

# Saving the result

In [14]:
save_dir = Path(r'\\allen\programs\mindscope\workgroups\learning\pilots\GCaMP8')
plane_save_fn = save_dir / 'gcamp8_plane_info_250418.csv'
gcamp_info.to_csv(plane_save_fn)

session_save_fn = save_dir / 'gcamp8_session_info_250418.csv'
gcamp_session_info.to_csv(session_save_fn)